# 03 – Phân tích lỗi (trên dự đoán THẬT của mô hình)

Đầu vào: `results/predictions/phobert_test_predictions.csv` (sinh ra bởi `src.train` hoặc `python -m src.evaluate`).
Mọi dự đoán ở đây đều lấy từ mô hình – **không** tự tạo/mô phỏng nhãn dự đoán.

In [ ]:
# Cho phép import src/ dù mở notebook từ thư mục notebooks/
import sys
from pathlib import Path
ROOT = Path.cwd().resolve()
while not (ROOT / "configs" / "config.yaml").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

from src.config import load_config, resolve
cfg = load_config()
print("Project root:", ROOT)

In [ ]:
import pandas as pd
pred_dir = resolve(cfg["results"]["predictions_dir"])
out_dir = resolve(cfg["results"]["dir"]) / "error_analysis"; out_dir.mkdir(parents=True, exist_ok=True)
names = cfg["labels"]

ph = pd.read_csv(pred_dir / "phobert_test_predictions.csv")
ph["confidence"] = ph[[f"prob_{i}" for i in names]].max(axis=1)
print(f"Test: {len(ph):,} câu | sai: {(~ph['correct']).sum():,} ({(~ph['correct']).mean():.2%})")
pd.crosstab(ph["label"].map(names), ph["pred"].map(names), margins=True,
            rownames=["Nhãn thật"], colnames=["Dự đoán"])

## 1. Các cặp nhầm lẫn phổ biến

In [ ]:
err = ph[~ph["correct"]].copy()
err["pair"] = err["label"].map(names) + " → " + err["pred"].map(names)
err["pair"].value_counts().to_frame("số câu")

In [ ]:
# Ví dụ cụ thể cho từng cặp nhầm, ưu tiên câu mô hình "tự tin mà sai"
pd.set_option("display.max_colwidth", 120)
(err.sort_values("confidence", ascending=False)
    .groupby("pair").head(5)[["pair", "text", "confidence"]]
    .sort_values("pair"))

## 2. Lỗi theo độ dài câu & độ tin cậy

In [ ]:
ph["n_words"] = ph["text"].str.split().str.len()
ph["len_bin"] = pd.cut(ph["n_words"], [0, 5, 10, 20, 40, 1000], labels=["1-5", "6-10", "11-20", "21-40", ">40"])
ph["conf_bin"] = pd.cut(ph["confidence"], [0, .5, .7, .9, 1.0])
display(ph.groupby("len_bin", observed=True)["correct"].agg(["count", "mean"]).rename(columns={"mean": "accuracy"}).round(3))
display(ph.groupby("conf_bin", observed=True)["correct"].agg(["count", "mean"]).rename(columns={"mean": "accuracy"}).round(3))

## 3. So sánh với baseline TF-IDF + LinearSVC

In [ ]:
bl_path = pred_dir / "baseline_tfidf_svc_balanced_test_predictions.csv"
if bl_path.exists():
    bl = pd.read_csv(bl_path)
    assert (bl["text"].values == ph["text"].values).all(), "Thứ tự câu không khớp"
    cmp_df = ph[["text", "label", "pred"]].rename(columns={"pred": "phobert_pred"})
    cmp_df["svc_pred"] = bl["pred"].values
    fixed  = cmp_df[(cmp_df.svc_pred != cmp_df.label) & (cmp_df.phobert_pred == cmp_df.label)]
    broken = cmp_df[(cmp_df.svc_pred == cmp_df.label) & (cmp_df.phobert_pred != cmp_df.label)]
    both   = cmp_df[(cmp_df.svc_pred != cmp_df.label) & (cmp_df.phobert_pred != cmp_df.label)]
    print(f"PhoBERT sửa đúng được: {len(fixed)} | PhoBERT làm sai thêm: {len(broken)} | cả hai cùng sai: {len(both)}")
    fixed.to_csv(out_dir / "phobert_fixed_vs_svc.csv", index=False, encoding="utf-8-sig")
    broken.to_csv(out_dir / "phobert_broken_vs_svc.csv", index=False, encoding="utf-8-sig")
    both.to_csv(out_dir / "both_wrong.csv", index=False, encoding="utf-8-sig")
    display(fixed[fixed.label == 1].head(10))
else:
    print("Chưa có dự đoán baseline – chạy: python -m src.baseline")

In [ ]:
# File tổng hợp toàn bộ câu sai (để nhóm đọc & ghi chú nguyên nhân thủ công)
cols = ["text", "label", "pred", "pair", "confidence"] + [f"prob_{i}" for i in names]
err[cols].assign(nguyen_nhan="").sort_values(["pair", "confidence"], ascending=[True, False]) \
    .to_csv(out_dir / "error_analysis.csv", index=False, encoding="utf-8-sig")
print("Đã lưu:", out_dir / "error_analysis.csv")

## 4. Nhận xét (nhóm điền sau khi đọc các ví dụ ở trên)

- Cặp nhầm nhiều nhất: …
- Đặc điểm câu Trung tính bị nhầm: …
- Hướng cải thiện: tách từ (word segmentation), thêm dữ liệu lớp Trung tính, …